In [1]:
import sqlite3
from functions.pred import *
from functions.xai import *
from functions.eval import *
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, jaccard_score

In [2]:
conn = sqlite3.connect('data/xs2a_db.sqlite3')
c = conn.cursor()

In [3]:
polarite_map = {1: 'positive', 2: 'neutral', 3: 'negative'}

In [4]:
comments = {}
c.execute('SELECT * FROM myApp_datasetcommentaire')
rows = c.fetchall()
for row in rows:
    comment_content = c.execute('SELECT * FROM myApp_commentaire WHERE id=' + str(row[0])).fetchall()[0][1]
    comments[row[0]] = {"commentaire": comment_content, "polarite": polarite_map[row[2]]}

In [5]:
annotations = []
c.execute('SELECT * FROM myApp_annotation')
rows = c.fetchall()
for row in rows:
    annotations.append({"id": row[3], "mot": row[1]})

In [6]:
# number of annotations words
len(annotations)

1980

In [7]:
annotated_comments = set()
for annotation in annotations:
    annotated_comments.add(annotation["id"])

In [8]:
# number of annotated comments
len(annotated_comments)

303

In [9]:
model_pred_col = "Camelbert-MSA"
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
# model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
# model_pred_col = "AraBert"

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_index = 0 if torch.cuda.is_available() else -1
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_attentions=True).to(device)
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device_index, top_k=None)
polarities = list(model.config.id2label.values())

Device set to use cuda:0


In [11]:
lime_num_samples = 100
shap_max_evals = 100
ig_n_steps = 50

In [12]:
lime_explainer = LimeExplainer(model_name, device, num_samples=lime_num_samples)
shap_explainer = ShapExplainer(model_name, device, max_evals=shap_max_evals)
ig_explainer = IgExplainer(model_name, device, n_steps=ig_n_steps)
dl_explainer = DeepLiftExplainer(model_name, device)
ensemble_explainer = EnsembleExplainer(mean=True, median=True)

In [13]:
def token_in_annotated_mots(token, annotated_mots):
    if token.startswith("##"):
        token = token[2:]
    for annotated_mot in annotated_mots:
        if token in annotated_mot:
            return True
    return False

In [14]:
for comment_id in annotated_comments:
    comment_text = comments[comment_id]["commentaire"]
    comment_text_preprocessed = remove_chaklas(comment_text)
    comments[comment_id]["commentaire"] = comment_text_preprocessed

In [15]:
df = pd.DataFrame(columns=["comment_id", "token", "polarity", "lime_weight", "shap_weight", "ig_weight", "deeplift_weight", "human_annot"])

In [16]:
for comment_id in tqdm(annotated_comments, desc="Explaining comments", unit="comment"):
    lime_res = lime_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])
    
    shap_res = shap_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])

    ig_res = ig_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])

    deeplift_res = dl_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])

    exai4_mean, exai4_median = ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res,
        ig_results=ig_res, dl_results=deeplift_res)

    exai3_mean, exai3_median = ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res,
        ig_results=ig_res, dl_results=None)

    exai2_mean, exai2_median = ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res,
        ig_results=None, dl_results=None)

    annotated_mots = {annotation["mot"]
        for annotation in annotations if annotation["id"] == comment_id}

    tokens = [r[1] for r in lime_res]

    for i, token in enumerate(tokens):
        human_annot = int(token_in_annotated_mots(token, annotated_mots))

        df = pd.concat([df,
            pd.DataFrame({
                "comment_id": [comment_id],
                "token": [token],
                "polarity": [comments[comment_id]["polarite"]],
                "lime_weight": [lime_res[i][2]],
                "shap_weight": [shap_res[i][2]],
                "ig_weight": [ig_res[i][2]],
                "deeplift_weight": [deeplift_res[i][2]],
                "exai4_mean": [exai4_mean[i][2]],
                "exai4_median": [exai4_median[i][2]],
                "exai3_mean": [exai3_mean[i][2]],
                "exai3_median": [exai3_median[i][2]],
                "exai2_mean": [exai2_mean[i][2]],
                "exai2_median": [exai2_median[i][2]],
                "human_annot": [human_annot]
            })], ignore_index=True)

Explaining comments:   0%|          | 0/303 [00:00<?, ?comment/s]

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\_utils\gradient.py:57: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\attr\_core\deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes will be removed
            after the attribution is finished
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\_utils\gradient.py:57: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\attr\_core\deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes w

In [17]:
df

,comment_id,token,polarity,lime_weight,shap_weight,ig_weight,deeplift_weight,human_annot,exai4_mean,exai4_median,exai3_mean,exai3_median,exai2_mean,exai2_median
0,1024,فندق,positive,0.929301,0.180437,0.376384,0.633807,0,0.529982,0.505096,0.495374,0.376384,0.554869,0.554869
1,1024,يناسب,positive,-1.0,0.261758,-0.081475,-1.0,0,-0.454929,-0.540737,-0.273239,-0.081475,-0.369121,-0.369121
2,1024,إمكان,positive,1.0,0.686973,-0.030491,0.003976,1,0.415114,0.345474,0.552161,0.686973,0.843487,0.843487
3,1024,##ياتك,positive,-0.322886,1.0,0.175941,0.15647,1,0.252381,0.166205,0.284352,0.175941,0.338557,0.338557
4,1024,إن,positive,-0.189936,0.603025,0.360297,-0.570814,0,0.050643,0.085180,0.257795,0.360297,0.206544,0.206544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16920,1023,##ه,negative,0.08565,0.603493,-1.0,-0.75304,1,-0.265974,-0.333695,-0.103619,0.085650,0.344572,0.344572
16921,1023,جدا,negative,-0.247186,0.298809,-0.142697,-0.811351,0,-0.225606,-0.194941,-0.030358,-0.142697,0.025812,0.025812
16922,1023,ومب,negative,-0.350319,1.0,-0.859514,-0.826283,1,-0.259029,-0.588301,-0.069944,-0.350319,0.324840,0.324840
16923,1023,##الغ,negative,-0.332303,0.178682,-0.834228,-0.887094,1,-0.468736,-0.583266,-0.329283,-0.332303,-0.076810,-0.076810


In [21]:
def top_tokens_elbow(token_scores):
    # Sort by importance descending
    sorted_token_weights = sorted(token_scores, key=lambda x: x[1], reverse=True)
    tokens = [x[0] for x in sorted_token_weights]
    scores = np.array([x[1] for x in sorted_token_weights])

    n = scores.size
    if n < 2:
        elbow_idx = 1
    else:
        x = np.arange(n, dtype=np.float64)
        x_mean = (n - 1) / 2.0
        y_mean = scores.mean()

        dx = x - x_mean
        slope = np.dot(dx, scores - y_mean) / np.dot(dx, dx)
        intercept = y_mean - slope * x_mean

        fitted = slope * x + intercept
        residuals = fitted - scores
        elbow_idx = np.argmax(residuals[1:]) + 1

    if elbow_idx is None:
        top_tokens = []  # No elbow detected
    else:
        threshold = scores[elbow_idx - 1]  # Adjust for 0-based index
        top_tokens = [tok for tok, score in zip(tokens, scores) if score >= threshold]
    return top_tokens

In [22]:
df.columns

Index(['comment_id', 'token', 'polarity', 'lime_weight', 'shap_weight',
       'ig_weight', 'deeplift_weight', 'human_annot', 'exai4_mean',
       'exai4_median', 'exai3_mean', 'exai3_median', 'exai2_mean',
       'exai2_median'],
      dtype='str')

In [25]:
df["lime_hard"] = 0
df["shap_hard"] = 0
df["ig_hard"] = 0
df["deeplift_hard"] = 0
df["exai4_mean_hard"] = 0
df["exai4_median_hard"] = 0
df["exai3_mean_hard"] = 0
df["exai3_median_hard"] = 0
df["exai2_mean_hard"] = 0

for comment_id in tqdm(annotated_comments, desc="Running elbow on explanations", unit="comment"):
    # Extract token-weight pairs for each method
    lime_token_scores = df[df["comment_id"] == comment_id][["token", "lime_weight"]].to_dict(orient="records")
    shap_token_scores = df[df["comment_id"] == comment_id][["token", "shap_weight"]].to_dict(orient="records")
    ig_token_scores = df[df["comment_id"] == comment_id][["token", "ig_weight"]].to_dict(orient="records")
    deeplift_token_scores = df[df["comment_id"] == comment_id][["token", "deeplift_weight"]].to_dict(orient="records")
    exai4_mean_scores = df[df["comment_id"] == comment_id][["token", "exai4_mean"]].to_dict(orient="records")
    exai4_median_scores = df[df["comment_id"] == comment_id][["token", "exai4_median"]].to_dict(orient="records")
    exai3_mean_scores = df[df["comment_id"] == comment_id][["token", "exai3_mean"]].to_dict(orient="records")
    exai3_median_scores = df[df["comment_id"] == comment_id][["token", "exai3_median"]].to_dict(orient="records")
    exai2_mean_scores = df[df["comment_id"] == comment_id][["token", "exai2_mean"]].to_dict(orient="records")

    # Convert to list of tuples instead of dicts, skipping NaNs
    lime_token_scores = [(item["token"], item["lime_weight"]) for item in lime_token_scores if not pd.isna(item["lime_weight"])]
    shap_token_scores = [(item["token"], item["shap_weight"]) for item in shap_token_scores if not pd.isna(item["shap_weight"])]
    ig_token_scores = [(item["token"], item["ig_weight"]) for item in ig_token_scores if not pd.isna(item["ig_weight"])]
    deeplift_token_scores = [(item["token"], item["deeplift_weight"]) for item in deeplift_token_scores if not pd.isna(item["deeplift_weight"])]
    exai4_mean_scores = [(item["token"], item["exai4_mean"]) for item in exai4_mean_scores if not pd.isna(item["exai4_mean"])]
    exai4_median_scores = [(item["token"], item["exai4_median"]) for item in exai4_median_scores if not pd.isna(item["exai4_median"])]
    exai3_mean_scores = [(item["token"], item["exai3_mean"]) for item in exai3_mean_scores if not pd.isna(item["exai3_mean"])]
    exai3_median_scores = [(item["token"], item["exai3_median"]) for item in exai3_median_scores if not pd.isna(item["exai3_median"])]
    exai2_mean_scores = [(item["token"], item["exai2_mean"]) for item in exai2_mean_scores if not pd.isna(item["exai2_mean"])]

    # Pass lists of tuples to top_tokens_elbow
    lime_top_tokens = top_tokens_elbow(lime_token_scores)
    shap_top_tokens = top_tokens_elbow(shap_token_scores)
    ig_top_tokens = top_tokens_elbow(ig_token_scores)
    deeplift_top_tokens = top_tokens_elbow(deeplift_token_scores)
    exai4_mean_top_tokens = top_tokens_elbow(exai4_mean_scores)
    exai4_median_top_tokens = top_tokens_elbow(exai4_median_scores)
    exai3_mean_top_tokens = top_tokens_elbow(exai3_mean_scores)
    exai3_median_top_tokens = top_tokens_elbow(exai3_median_scores)
    exai2_mean_top_tokens = top_tokens_elbow(exai2_mean_scores)

    # Mark top tokens in the DataFrame
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(lime_top_tokens)), "lime_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(shap_top_tokens)), "shap_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(ig_top_tokens)), "ig_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(deeplift_top_tokens)), "deeplift_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai4_mean_top_tokens)), "exai4_mean_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai4_median_top_tokens)), "exai4_median_hard"] = 1 
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai3_mean_top_tokens)), "exai3_mean_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai3_median_top_tokens)), "exai3_median_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai2_mean_top_tokens)), "exai2_mean_hard"] = 1

Running elbow on explanations:   0%|          | 0/303 [00:00<?, ?comment/s]

In [42]:
df.to_csv("data/plausibility/xai_res_with_annotations.csv", index=False)

In [2]:
df = pd.read_csv("data/plausibility/xai_res_with_annotations.csv")

In [3]:
methods_hard_cols = ["lime_hard", "shap_hard", "ig_hard", "deeplift_hard", "exai4_mean_hard", "exai4_median_hard", "exai3_mean_hard", "exai3_median_hard", "exai2_mean_hard"]

In [4]:
df["human_annot"] = df["human_annot"].astype(int)

In [5]:
plausibility_metrics = {}
for method_col in methods_hard_cols:
    accuracy = accuracy_score(df["human_annot"], df[method_col])
    precision = precision_score(df["human_annot"], df[method_col])
    recall = recall_score(df["human_annot"], df[method_col])
    f1 = f1_score(df["human_annot"], df[method_col])
    jaccard = jaccard_score(df["human_annot"], df[method_col])
    plausibility_metrics[method_col] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "jaccard": jaccard
    }

In [6]:
plausibility_metrics_df = pd.DataFrame(plausibility_metrics).T
plausibility_metrics_df

,accuracy,precision,recall,f1,jaccard
lime_hard,0.395096,0.259691,0.735470,0.383847,0.237507
shap_hard,0.493648,0.285120,0.647832,0.395968,0.246858
ig_hard,0.355628,0.265357,0.856780,0.405214,0.254087
deeplift_hard,0.316337,0.261331,0.913515,0.406402,0.255022
exai4_mean_hard,0.362659,0.267431,0.855397,0.407470,0.255864
exai4_median_hard,0.364195,0.266923,0.848478,0.406093,0.254778
exai3_mean_hard,0.413530,0.275068,0.788284,0.407827,0.256145
exai3_median_hard,0.386647,0.269328,0.813884,0.404725,0.253702
exai2_mean_hard,0.430428,0.276617,0.757380,0.405232,0.254101
